# 投资组合排序法完整教程：从 10 只股票到真实因子检验

## 📚 教学目标
1. 理解因子投资中**投资组合排序法**的核心逻辑与动机
2. 在 **10 只股票的微型数据集**上手算每一步，建立直觉
3. 掌握**单因子排序** (Single Sort) 的完整 Python 实现
4. 理解**双重排序** (Double Sorts) 的独立排序与依赖排序
5. 使用**多期时间序列数据**完成 T 检验与单调性检验

**参考书**：《因子投资：方法与实践》（石川）第 2 章
**教学策略**：先用极小数据集让你“看见”每一步计算，再扩展到真实规模

---

## 1. 为什么需要投资组合排序法？

### 🎯 一个直觉问题

假设有人告诉你：“小市值股票未来收益更高”。你会怎么验证？

最自然的想法：
1. 把所有股票按市值**从小到大排序**
2. 分成几组（比如 5 组）
3. 看看小市值那组的收益是不是真的比大市值那组高

这就是**投资组合排序法（Portfolio Sorts）**的核心思想！

### 📖 书中的定义

> 投资组合排序法是实证资产定价中最基础也是最常用的工具。其核心逻辑是：
> 如果某个因子是有效的，那么按因子值排序后，高因子值组与低因子值组的未来收益应存在显著差异。

### 📐 因子模拟投资组合 (Factor Mimicking Portfolio, FMP)

排序法的最终目标是构建一个**因子模拟投资组合 (FMP)**：

$$\text{FMP} = R_{\text{Long}} - R_{\text{Short}}$$

```
"因子"是抽象概念 ──→ 你无法直接"买入市值因子"
                 ──→ 但你可以构建一个组合来"模拟"它
                 ──→ 做多小市值 + 做空大市值 = SMB 因子
```

| 经典因子 | 做多 | 做空 | FMP 名称 |
|---------|------|------|----------|
| 市值因子 | 小市值股 | 大市值股 | SMB (Small Minus Big) |
| 价值因子 | 高 B/M 股 | 低 B/M 股 | HML (High Minus Low) |
| 动量因子 | 过去涨幅高的 | 过去涨幅低的 | WML (Winners Minus Losers) |

接下来，我们从最小的例子开始，一步步构建这个过程。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置中文字体和绘图风格
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 从 10 只股票开始：手算单因子排序

### 🎯 场景

假设整个市场只有 **10 只股票**。我们有每只股票的：
- **市值**（排序变量 / 因子）
- **下一期收益率**（我们要预测的）

**假设**：市值是**负向因子**（小市值收益更高），我们来验证这个假设。

In [ ]:
# ========== 构造 10 只股票的微型数据集 ==========
# 手动构造，让数据直觉清晰
mini_data = pd.DataFrame({
    '股票': ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'],
    '市值(亿)': [5, 12, 18, 25, 35, 48, 60, 78, 90, 120],
    '下期收益率(%)': [8.2, 6.5, 5.1, 4.8, 3.2, 2.5, 1.8, 0.5, -0.3, -1.2]
})

print("📋 原始数据（已按市值从小到大排列）：")
print(mini_data.to_string(index=False))
print(f"\n💡 观察：市值越小，收益率越高 —— 这就是我们要验证的 Size Effect")

### 📐 步骤 1：排序

按排序变量（市值）从小到大排列所有股票。

上面的数据已经排好了，实际操作中这是第一步。

In [ ]:
# ========== 步骤 1: 排序 ==========
mini_sorted = mini_data.sort_values('市值(亿)').reset_index(drop=True)

print("📊 步骤 1: 按市值从小到大排序")
print(mini_sorted.to_string(index=False))
print(f"\n✅ 排序完成。数据已经是排好的，但在真实研究中这一步不可省略。")

### 📐 步骤 2：分组

将排好序的股票**等分为 $N$ 组**。常见分组方式：

- $N = 5$：五分位数 (Quintiles) —— 最常用
- $N = 10$：十分位数 (Deciles)
- $N = 3$：三分位数 (Terciles)

分位数断点 (Breakpoints)：

$$BP_k = P_{k \times (100/N)}, \quad k = 1, 2, \ldots, N-1$$

这里 10 只股票分 5 组，每组 2 只：

```
Q1 (最小市值): 股票 A, B
Q2:            股票 C, D
Q3:            股票 E, F
Q4:            股票 G, H
Q5 (最大市值): 股票 I, J
```

In [ ]:
# ========== 步骤 2: 分组 ==========
N_GROUPS = 5
labels = [f'Q{i}' for i in range(1, N_GROUPS + 1)]

# pd.qcut 按分位数自动等分
mini_sorted['分组'] = pd.qcut(
    mini_sorted['市值(亿)'],
    q=N_GROUPS,
    labels=labels
)

print("📊 步骤 2: 分组结果")
print("─" * 50)
for q in labels:
    group = mini_sorted[mini_sorted['分组'] == q]
    stocks = ', '.join(group['股票'].values)
    caps = ', '.join([f"{v}亿" for v in group['市值(亿)'].values])
    print(f"  {q}: [{stocks}]  市值 = [{caps}]")

print(f"\n💡 pd.qcut(data, q=5) 会自动找到分位数断点，保证每组数量尽量相等")

### 📐 步骤 3：计算各组平均收益率

**等权平均 (Equal-Weighted)**：组内每只股票权重相同

$$R_{Q_i} = \frac{1}{n_i} \sum_{j \in Q_i} r_j$$

我们来手算 Q1 的收益率：

$$R_{Q1} = \frac{r_A + r_B}{2} = \frac{8.2\% + 6.5\%}{2} = 7.35\%$$

In [ ]:
# ========== 步骤 3: 计算各组平均收益率 ==========
print("📊 步骤 3: 逐组计算等权平均收益率")
print("─" * 60)

group_returns = {}
for q in labels:
    group = mini_sorted[mini_sorted['分组'] == q]
    stocks = group['股票'].values
    rets = group['下期收益率(%)'].values
    avg_ret = rets.mean()
    group_returns[q] = avg_ret

    # 展示计算过程
    ret_str = ' + '.join([f'{r}%' for r in rets])
    print(f"  {q} ({', '.join(stocks)}):")
    print(f"     R_{q} = ({ret_str}) / {len(rets)}")
    print(f"          = {sum(rets):.1f}% / {len(rets)}")
    print(f"          = {avg_ret:.2f}%")
    print()

portfolio_rets = pd.Series(group_returns)
print("📈 汇总:")
for q, r in group_returns.items():
    bar = '█' * int(max(0, r) * 3)
    print(f"  {q}: {r:>6.2f}%  {bar}")

### 📐 步骤 4：构建多空组合 (Long-Short Portfolio)

#### 💡 核心概念：为什么要构建多空组合？

**问题**：光看 Q1 收益高就能说明市值因子有效吗？

**不能！** 因为 Q1 的高收益可能来自：
- 整个市场的上涨（牛市效应）
- 随机噪声
- 其他因子的影响

**解决方案**：构建**零成本多空组合**

$$R_{\text{spread}} = R_{\text{Long}} - R_{\text{Short}} = R_{Q1} - R_{Q5}$$

做多与做空的金额相等（各 $1），所以：
- **零净投入**：不需要额外资金
- **对冲市场风险**：牛市涨跌对多空两端影响相似，相减后抵消
- **剥离因子纯效应**：剩下的就是“小市值vs大市值”的纯收益差异

#### 📖 书中解释

> 因子模拟投资组合的核心在于**资金标准化（Dollar Neutrality）**：做多 $1 的高暴露组，做空 $1 的低暴露组，构建零净投入的组合。

In [ ]:
# ========== 步骤 4: 构建多空组合 ==========
# 市值是负向因子 → 做多小市值(Q1)，做空大市值(Q5)
long_ret = group_returns['Q1']
short_ret = group_returns['Q5']
spread = long_ret - short_ret

print("📊 步骤 4: 构建多空组合 (SMB)")
print("─" * 50)
print(f"  做多 Q1 (小市值):  R_long  = {long_ret:.2f}%")
print(f"  做空 Q5 (大市值):  R_short = {short_ret:.2f}%")
print(f"")
print(f"  📐 Spread = R_long − R_short")
print(f"           = {long_ret:.2f}% − ({short_ret:.2f}%)")
print(f"           = {spread:.2f}%")
print(f"")
print(f"  💡 解读: 做多小市值、做空大市值，单期收益为 {spread:.2f}%")
print(f"     但这只是 1 期的结果，我们还不能下结论！")
print(f'     需要多期数据 + T检验才能判断这个收益是否"显著不为零"')

In [ ]:
# ========== 可视化：单因子排序结果 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：各组收益率柱状图 ---
ax1 = axes[0]
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']
bars = ax1.bar(labels, [group_returns[q] for q in labels],
                color=colors, edgecolor='black', alpha=0.85)

# 标注数值
for bar, q in zip(bars, labels):
    height = bar.get_height()
    va = 'bottom' if height >= 0 else 'top'
    offset = 0.1 if height >= 0 else -0.1
    ax1.text(bar.get_x() + bar.get_width()/2., height + offset,
             f'{group_returns[q]:.2f}%', ha='center', va=va, fontweight='bold')

ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Portfolio (Q1=Smallest, Q5=Largest)', fontsize=11)
ax1.set_ylabel('Average Return (%)', fontsize=11)
ax1.set_title('Single Sort: Portfolio Returns', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 右图：散点图 + 趋势线 ---
ax2 = axes[1]
ax2.scatter(mini_sorted['市值(亿)'], mini_sorted['下期收益率(%)'],
             c=mini_sorted['分组'].cat.codes, cmap='RdYlGn_r',
             s=100, edgecolors='black', zorder=5)

# 添加股票标签
for _, row in mini_sorted.iterrows():
    ax2.annotate(row['股票'], (row['市值(亿)'], row['下期收益率(%)']),
                textcoords="offset points", xytext=(5, 5), fontsize=9)

# 趋势线
z = np.polyfit(mini_sorted['市值(亿)'], mini_sorted['下期收益率(%)'], 1)
p = np.poly1d(z)
x_line = np.linspace(mini_sorted['市值(亿)'].min(), mini_sorted['市值(亿)'].max(), 100)
ax2.plot(x_line, p(x_line), 'r--', linewidth=2, alpha=0.7, label=f'Trend (slope={z[0]:.4f})')

ax2.set_xlabel('Market Cap (Billion)', fontsize=11)
ax2.set_ylabel('Next-Period Return (%)', fontsize=11)
ax2.set_title('Market Cap vs Return: Negative Relationship', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 左图呈现清晰的递减阶梯：Q1 > Q2 > Q3 > Q4 > Q5")
print(f"   右图呈现负向线性关系：市值越大，收益越低")
print(f"   但这仅是 1 期数据，统计上还不够 —— 下一节我们用多期数据检验")

---

## 3. 多期数据下的单因子排序与 T 检验

### 🎯 关键问题

上一节我们只看了 1 期数据。但因子投资研究必须回答的核心问题是：

> **因子溢价（Spread）在统计上是否显著不为零？**

要回答这个问题，需要：
1. **多期数据**：每个月（或每周）重新排序、分组、计算 Spread
2. **时间序列**：收集一串 Spread 值，形成时间序列
3. **T 检验**：对这串 Spread 做单样本 T 检验（$H_0: \mu = 0$）

### 📐 完整流程

$$t = \frac{\bar{R}_{\text{spread}}}{\sigma_{\text{spread}} / \sqrt{T}}$$

```
每个月 t:
  1. 获取所有股票的因子值和收益率
  2. 按因子值排序 → 分组
  3. 计算各组等权平均收益
  4. Spread_t = R_Q1,t − R_Q5,t

收集 T 个月的 Spread:
  Spread = [Spread_1, Spread_2, ..., Spread_T]

T 检验:
  H₀: E[Spread] = 0
  t统计量 = Spread均值 / (Spread标准差 / √T)
```

In [ ]:
# ========== 模拟 60 个月的多期数据 ==========
np.random.seed(42)
N_STOCKS = 200     # 每月 200 只股票
N_MONTHS = 60      # 60 个月（5 年）
N_GROUPS = 5       # 五分位数

# 存储每月各组的收益率
monthly_group_returns = {f'Q{i}': [] for i in range(1, N_GROUPS + 1)}
monthly_spreads = []  # 每月的多空 Spread

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS} 组")
print(f"")
print(f"🔄 开始逐月排序...")
print(f"")

for t in range(N_MONTHS):
    # --- 每月生成新的截面数据 ---
    # 市值（对数正态分布）
    market_cap = np.random.lognormal(mean=10, sigma=1, size=N_STOCKS)

    # 收益率：包含市值因子效应 + 噪声
    # 核心：小市值（低 log_cap）→ 高收益
    log_cap = np.log(market_cap)
    # 因子效应：log_cap 每增加1，收益率下降约 0.3%
    factor_effect = -0.3 * (log_cap - log_cap.mean())
    noise = np.random.normal(0, 5, N_STOCKS)  # 较大的噪声
    returns = 1.0 + factor_effect + noise  # 基础收益 1% + 因子效应 + 噪声

    # 构建 DataFrame
    month_df = pd.DataFrame({
        'market_cap': market_cap,
        'return': returns
    })

    # --- 排序分组 ---
    month_df['group'] = pd.qcut(
        month_df['market_cap'], q=N_GROUPS,
        labels=[f'Q{i}' for i in range(1, N_GROUPS+1)]
    )

    # --- 计算各组等权平均收益 ---
    group_ret = month_df.groupby('group')['return'].mean()
    for q in monthly_group_returns:
        monthly_group_returns[q].append(group_ret[q])

    # --- 计算 Spread ---
    spread_t = group_ret['Q1'] - group_ret['Q5']  # 小市值 - 大市值
    monthly_spreads.append(spread_t)

    # 打印前 3 个月的详细过程
    if t < 3:
        print(f"  月份 {t+1}:")
        for q in [f'Q{i}' for i in range(1, N_GROUPS+1)]:
            print(f"    {q}: 平均收益 = {group_ret[q]:>6.2f}%")
        print(f"    Spread (Q1−Q5) = {group_ret['Q1']:.2f}% − {group_ret['Q5']:.2f}% = {spread_t:.2f}%")
        print()

print(f"  ... (省略第 4~{N_MONTHS} 月) ...")
print(f"\n✅ 完成！共获得 {len(monthly_spreads)} 个月的 Spread 数据")

### 📐 T 检验：因子预期收益率是否显著不为零？

#### 假设设定

- **H₀（原假设）**：$E[R_{spread}] = 0$，即因子无效，多空组合的预期收益为零
- **H₁（备择假设）**：$E[R_{spread}] \neq 0$，即因子有效

#### 计算公式

$$t = \frac{\bar{R}_{spread}}{\text{SE}} = \frac{\bar{R}_{spread}}{\sigma_{spread} / \sqrt{T}}$$

其中：
- $\bar{R}_{spread}$ = Spread 的时间序列均值
- $\sigma_{spread}$ = Spread 的时间序列标准差
- $T$ = 期数（月数）
- $\text{SE}$ = 标准误

In [ ]:
# ========== T 检验：逐步计算 ==========
spreads = np.array(monthly_spreads)

# --- 步骤 1: Spread 均值 ---
spread_mean = spreads.mean()
print(f"📊 步骤 1: 计算 Spread 均值")
print(f"  R̄_spread = (Spread_1 + Spread_2 + ... + Spread_{N_MONTHS}) / {N_MONTHS}")
print(f"           = {spreads.sum():.4f} / {N_MONTHS}")
print(f"           = {spread_mean:.4f}%")
print()

# --- 步骤 2: Spread 标准差 ---
spread_std = spreads.std(ddof=1)  # 样本标准差
print(f"📊 步骤 2: 计算 Spread 标准差")
print(f"  σ_spread = √[Σ(Spread_t - R̄)² / (T-1)]")
print(f"           = {spread_std:.4f}%")
print()

# --- 步骤 3: 标准误 ---
spread_se = spread_std / np.sqrt(N_MONTHS)
print(f"📊 步骤 3: 计算标准误")
print(f"  SE = σ_spread / √T")
print(f"     = {spread_std:.4f} / √{N_MONTHS}")
print(f"     = {spread_std:.4f} / {np.sqrt(N_MONTHS):.4f}")
print(f"     = {spread_se:.4f}%")
print()

# --- 步骤 4: T 统计量 ---
t_stat = spread_mean / spread_se
print(f"📊 步骤 4: 计算 T 统计量")
print(f"  t = R̄_spread / SE")
print(f"    = {spread_mean:.4f} / {spread_se:.4f}")
print(f"    = {t_stat:.4f}")
print()

# --- 步骤 5: P 值 ---
df = N_MONTHS - 1
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=df))
print(f"📊 步骤 5: 计算 P 值 (双尾检验)")
print(f"  自由度 df = {N_MONTHS} - 1 = {df}")
print(f"  P 值 = 2 × P(T > |{t_stat:.4f}|)")
print(f"       = {p_value:.6f}")
print()

# --- 步骤 6: 结论 ---
alpha = 0.05
print(f"📊 步骤 6: 显著性判断")
print(f"  显著性水平 α = {alpha}")
if p_value < alpha:
    print(f"  ✅ P = {p_value:.6f} < {alpha}")
    print(f"  ✅ |t| = {abs(t_stat):.2f} > 2 (经验法则)")
    print(f"  ✅ 拒绝 H₀：因子预期收益率显著不为零")
    print(f"  ✅ 结论：市值因子（SMB）是有效的！")
else:
    print(f"  ❌ P = {p_value:.6f} ≥ {alpha}")
    print(f"  ❌ 不能拒绝 H₀：没有足够证据说明因子有效")

In [ ]:
# ========== 用 scipy 验证 ==========
t_scipy, p_scipy = stats.ttest_1samp(spreads, 0)

print("🔬 scipy.stats.ttest_1samp 验证:")
print(f"  手算 T 统计量 = {t_stat:.6f}")
print(f"  scipy T 统计量 = {t_scipy:.6f}")
print(f"  手算 P 值     = {p_value:.6f}")
print(f"  scipy P 值    = {p_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

In [ ]:
# ========== 可视化 T 检验 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: Spread 时间序列 ---
ax1 = axes[0]
months = range(1, N_MONTHS + 1)
ax1.bar(months, spreads, color=['#2ecc71' if s > 0 else '#e74c3c' for s in spreads],
        alpha=0.7, edgecolor='none')
ax1.axhline(y=spread_mean, color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {spread_mean:.2f}%')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('Spread (%)', fontsize=11)
ax1.set_title('Monthly Spread (Q1-Q5)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: Spread 分布 ---
ax2 = axes[1]
ax2.hist(spreads, bins=15, edgecolor='black', alpha=0.7, color='steelblue', density=True)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='$H_0$: $\mu$=0')
ax2.axvline(x=spread_mean, color='blue', linestyle='--', linewidth=2,
            label=f'Sample Mean = {spread_mean:.2f}%')
# 叠加正态拟合
x_fit = np.linspace(spreads.min() - 1, spreads.max() + 1, 100)
ax2.plot(x_fit, stats.norm.pdf(x_fit, spread_mean, spread_std), 'b-', linewidth=2)
ax2.set_xlabel('Spread (%)', fontsize=11)
ax2.set_ylabel('Probability Density', fontsize=11)
ax2.set_title('Spread Distribution', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

# --- 图3: T 分布与 T 值位置 ---
ax3 = axes[2]
x_t = np.linspace(-5, 5, 1000)
t_dist = stats.t.pdf(x_t, df=df)
ax3.plot(x_t, t_dist, 'b-', linewidth=2, label=f'T distribution (df={df})')
ax3.fill_between(x_t, t_dist, where=(x_t >= abs(t_stat)), color='red', alpha=0.4)
ax3.fill_between(x_t, t_dist, where=(x_t <= -abs(t_stat)), color='red', alpha=0.4,
                 label=f'P-value region = {p_value:.4f}')
ax3.axvline(x=t_stat, color='red', linestyle='--', linewidth=2,
            label=f't = {t_stat:.2f}')
ax3.set_xlabel('T Value', fontsize=11)
ax3.set_ylabel('Probability Density', fontsize=11)
ax3.set_title('T-Test Visualization', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：绿色=正Spread（小市值赢），红色=负Spread（大市值赢）")
print(f"       蓝色虚线 = 平均Spread = {spread_mean:.2f}%，偏离零线说明因子有效")
print(f"  图2：Spread 的分布集中在正值区域，远离 H₀ 假设的 0")
print(f"  图3：红色阴影面积 = P值 = {p_value:.4f}，非常小 → 拒绝 H₀")

### 📐 单调性检验 (Monotonicity Test)

T 检验只告诉我们 "Q1 和 Q5 有差异"，但一个好的因子还应该满足：

> **从 Q1 到 Q5，收益率应该单调递减（或递增）。**

#### 📖 书中原文

> 一个好的因子应该能够解释个股超额收益的截面差异。因此，依照排序变量高低得到的 L 个投资组合的收益率应该表现出很好的**单调性**。

#### 检验方法：Spearman 秩相关系数

$$\rho_s = \text{Corr}(\text{组别序号}, \text{组别收益率})$$

- $\rho_s = -1$：完美单调递减（如市值因子）
- $\rho_s = +1$：完美单调递增（如价值因子）
- $\rho_s \approx 0$：无单调关系

In [ ]:
# ========== 单调性检验 ==========
# 计算各组的时间序列平均收益
avg_group_rets = {q: np.mean(monthly_group_returns[q]) for q in monthly_group_returns}

print("📊 各组平均收益率（60 个月平均）：")
print("─" * 40)
for q in [f'Q{i}' for i in range(1, N_GROUPS + 1)]:
    bar = '█' * int(max(0, avg_group_rets[q]) * 5)
    print(f"  {q}: {avg_group_rets[q]:>6.3f}%  {bar}")

# Spearman 秩相关
group_ranks = np.arange(1, N_GROUPS + 1)
group_ret_values = np.array([avg_group_rets[f'Q{i}'] for i in range(1, N_GROUPS + 1)])
spearman_corr, spearman_p = stats.spearmanr(group_ranks, group_ret_values)

print(f"\n📐 Spearman 秩相关系数计算：")
print(f"  组别序号:    {list(group_ranks)}")
print(f"  组别收益率:  {[f'{r:.3f}' for r in group_ret_values]}")
print(f"")
print(f"  ρ_s = {spearman_corr:.4f}")
print(f"  P 值 = {spearman_p:.6f}")
print(f"")
if spearman_corr < -0.8:
    print(f"  ✅ ρ_s = {spearman_corr:.2f} ≈ −1，呈现强单调递减")
    print(f"  ✅ 市值从小到大，收益率单调递减，因子质量优秀")
elif spearman_corr < -0.5:
    print(f"  ⚠️ ρ_s = {spearman_corr:.2f}，单调递减但不够完美")
else:
    print(f"  ❌ ρ_s = {spearman_corr:.2f}，单调性不佳")

In [ ]:
# ========== 可视化：单调性 ==========
fig, ax = plt.subplots(figsize=(10, 6))

group_names = [f'Q{i}' for i in range(1, N_GROUPS + 1)]
rets = [avg_group_rets[q] for q in group_names]
colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, N_GROUPS))

bars = ax.bar(group_names, rets, color=colors, edgecolor='black', alpha=0.85)

# 标注数值
for bar, r in zip(bars, rets):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{r:.3f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 添加趋势线
ax.plot(group_names, rets, 'ko--', linewidth=2, markersize=8, zorder=5)

ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xlabel('Portfolio (Q1=Smallest Cap → Q5=Largest Cap)', fontsize=12)
ax.set_ylabel('Average Monthly Return (%)', fontsize=12)
ax.set_title(f'Monotonicity Test: Spearman \u03c1 = {spearman_corr:.3f} (P = {spearman_p:.4f})',
             fontsize=14, fontweight='bold')

# 添加注释框
textstr = f'Spread (Q1-Q5) = {spread_mean:.3f}%\nt = {t_stat:.2f}, P = {p_value:.4f}\nSpearman \u03c1 = {spearman_corr:.3f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.98, 0.98, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', horizontalalignment='right', bbox=props)

ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 单因子排序小结

| 检验项目 | 方法 | 核心指标 | 判断标准 |
|---------|------|---------|----------|
| 因子溢价显著性 | T 检验 | t 统计量 | \|t\| > 2 (经验法则) |
| 因子单调性 | Spearman 秩相关 | ρ_s | \|ρ\| 接近 1 |

**两步判断法**：
1. T 检验通过 → 因子溢价显著
2. 单调性通过 → 因子质量优秀

只通过 T 检验但单调性差的因子，可能只是极端组差异大，中间组混乱 —— 这种因子在实际投资中不可靠。

---

## 4. 双重排序法 (Double Sorts)

### 🎯 为什么需要双重排序？

**问题**：如果两个因子之间存在相关性，单因子排序的结论可能不可靠。

**例子**：
- 小市值股票往往 B/M 比率更高（小公司的市场价格低，账面价值相对较高）
- 如果你只按 B/M 排序，发现 "高 B/M 收益高"
- 但这个收益到底来自**价值因子**还是**市值因子**？分不清！

**解决方案**：双重排序 —— 先控制一个因子，再检验另一个因子

```
单因子排序的问题：
  按 B/M 排序 → 高 B/M 组收益高
  但高 B/M 组中小市值股票多 → 收益可能来自市值效应

双重排序的解决：
  先按市值分组 → 控制住市值效应
  再在每个市值组内按 B/M 排序 → 剥离出纯 B/M 效应
```

### 4.1 先用 25 只股票看清楚机制

我们构造一个 **25 只股票**的迷你数据集。

- **因子 A**：市值 (Size)
- **因子 B**：账面市值比 (B/M)
- **关键设定**：Size 和 B/M 之间有**负相关** (相关系数 ≈ −0.6)，模拟真实市场特征

In [ ]:
# ========== 构造 25 只股票的双因子数据 ==========
np.random.seed(42)
N = 25

# 市值：均匀分布在 10~100 亿
size = np.linspace(10, 100, N)

# B/M：与市值负相关 + 噪声
# 小市值 → 高 B/M，大市值 → 低 B/M
btm = 2.0 - 0.015 * size + np.random.normal(0, 0.2, N)
btm = np.clip(btm, 0.3, 2.5)  # 限制合理范围

# 收益率：同时受 Size (负向) 和 B/M (正向) 影响
ret = 5.0 - 0.03 * size + 2.0 * btm + np.random.normal(0, 1.5, N)

df_demo = pd.DataFrame({
    '股票': [f'S{i+1:02d}' for i in range(N)],
    'Size': np.round(size, 1),
    'B2M': np.round(btm, 3),
    'Ret': np.round(ret, 2)
})

print("📋 25 只股票数据集：")
print(df_demo.to_string(index=False))

corr = df_demo['Size'].corr(df_demo['B2M'])
print(f"\n📐 Size 与 B/M 的相关系数: {corr:.3f}")
print(f"💡 负相关！小市值股票倾向于有高 B/M —— 这就是我们需要双重排序的原因")

In [ ]:
# ========== 可视化两因子关系 ==========
fig, ax = plt.subplots(figsize=(8, 6))

scatter = ax.scatter(df_demo['Size'], df_demo['B2M'],
                      c=df_demo['Ret'], cmap='RdYlGn', s=100,
                     edgecolors='black', zorder=5)

# 标注股票名
for _, row in df_demo.iterrows():
    ax.annotate(row['股票'], (row['Size'], row['B2M']),
                textcoords="offset points", xytext=(4, 4), fontsize=7)

plt.colorbar(scatter, label='Return (%)')

# 趋势线
z = np.polyfit(df_demo['Size'], df_demo['B2M'], 1)
x_line = np.linspace(df_demo['Size'].min(), df_demo['Size'].max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), 'r--', linewidth=2, alpha=0.7)

ax.set_xlabel('Market Cap (Billion)', fontsize=12)
ax.set_ylabel('Book-to-Market (B/M)', fontsize=12)
ax.set_title(f'Size vs B/M (Correlation = {corr:.3f})', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 颜色越绿 = 收益越高；可以看到左上角（小市值 + 高B/M）最绿")
print("   问题是：这些股票的高收益到底是 Size 贡献还是 B/M 贡献？")

### 4.2 独立双重排序 (Independent Sort)

#### 📐 操作步骤

```
独立排序：两个因子各自独立分组，然后取交集

步骤 1：按 Size 分 5 组 → S1, S2, S3, S4, S5
步骤 2：按 B/M  分 5 组 → B1, B2, B3, B4, B5
步骤 3：取交集 → 25 个组合 (S1∩B1, S1∩B2, ..., S5∩B5)

注意：两步分组完全独立，互不影响
```

#### ⚠️ 独立排序的潜在问题

当两个因子存在相关性时，某些交叉组合可能**没有股票**（空桶问题）！

```
例如：
  Size 和 B/M 负相关
  → 大市值(S5) + 高B/M(B5) 这个组合可能找不到股票
  → 小市值(S1) + 低B/M(B1) 这个组合也可能找不到股票
```

In [ ]:
# ========== 独立排序 ==========
# 步骤 1: 按 Size 独立分 5 组
df_demo['Size_Group'] = pd.qcut(df_demo['Size'], 5, labels=['S1','S2','S3','S4','S5'])

# 步骤 2: 按 B/M 独立分 5 组
df_demo['B2M_Group'] = pd.qcut(df_demo['B2M'], 5, labels=['B1','B2','B3','B4','B5'])

# 展示分组结果
print("📊 独立排序分组结果：")
print("─" * 60)
print(df_demo[['股票', 'Size', 'B2M', 'Ret', 'Size_Group', 'B2M_Group']].to_string(index=False))

# 步骤 3: 统计每个交叉组合的股票数量
print("\n📊 交叉组合中的股票数量：")
count_matrix = df_demo.groupby(['Size_Group', 'B2M_Group']).size().unstack(fill_value=0)
print(count_matrix)

n_empty = (count_matrix == 0).sum().sum()
n_total = count_matrix.shape[0] * count_matrix.shape[1]
print(f"\n⚠️ 共 {n_total} 个组合，其中 {n_empty} 个为空桶 ({n_empty/n_total*100:.0f}%)")
print(f"💡 空桶太多会导致分析结果不可靠 —— 这就是独立排序的主要缺陷")

In [ ]:
# ========== 可视化：空桶问题 ==========
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图：股票数量热力图 ---
ax1 = axes[0]
sns.heatmap(count_matrix, annot=True, fmt='d', cmap='YlOrRd',
             linewidths=1, ax=ax1, cbar_kws={'label': 'Stock Count'})
ax1.set_title('Independent Sort: Stocks per Cell', fontsize=13, fontweight='bold')
ax1.set_xlabel('B/M Group', fontsize=11)
ax1.set_ylabel('Size Group', fontsize=11)

# --- 右图：散点图标注分组 ---
ax2 = axes[1]
for _, row in df_demo.iterrows():
    ax2.scatter(row['Size'], row['B2M'], s=100, edgecolors='black', zorder=5,
                c='steelblue')
    ax2.annotate(f"{row['Size_Group']},{row['B2M_Group']}",
                 (row['Size'], row['B2M']),
                textcoords="offset points", xytext=(5, 5), fontsize=7)

# 画分组线
for q in [0.2, 0.4, 0.6, 0.8]:
    ax2.axvline(x=df_demo['Size'].quantile(q), color='red', alpha=0.3, linestyle='--')
    ax2.axhline(y=df_demo['B2M'].quantile(q), color='blue', alpha=0.3, linestyle='--')

ax2.set_xlabel('Size', fontsize=11)
ax2.set_ylabel('B/M', fontsize=11)
ax2.set_title('Independent Sort: Group Assignment', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("\n💡 左图：红色越深 = 股票越多。大量 0（白色）表示空桶")
print("   右图：红色竖线 = Size 断点，蓝色横线 = B/M 断点")
print("         因为负相关，股票集中在左上→右下的对角线上，角落处没有股票")

### 4.3 依赖双重排序 (Conditional / Dependent Sort)

#### 📐 操作步骤

```
依赖排序：先按控制变量分组，再在每组内部按目标变量分组

步骤 1：按 Size 分 5 组 → S1, S2, S3, S4, S5 (每组 5 只)
步骤 2：在每个 Size 组内部，按 B/M 再分 5 组
        S1 内部: 5 只股票 → 按 B/M 排序 → B1, B2, B3, B4, B5 (每组 1 只)
        S2 内部: 5 只股票 → 按 B/M 排序 → B1, B2, B3, B4, B5 (每组 1 只)
        ...
```

#### 💡 为什么依赖排序能解决空桶问题？

因为每个 Size 组内**一定有**从低到高的 B/M 股票（只是"相对"的高低）。即使 S5（大市值）组的 B/M 整体偏低，组内仍然有"相对高 B/M"和"相对低 B/M"的股票。

#### 📖 书中要点

> 依赖排序能有效控制控制变量的影响，保证每个交叉分组中都有足够的股票。但代价是：第二个因子的分组不再是全市场范围的等分位，而是组内等分位。

#### 两种排序对比

| 特性 | 独立排序 | 依赖排序 |
|------|---------|----------|
| 空桶问题 | ⚠️ 因子相关时严重 | ✅ 不存在 |
| 对称性 | ✅ 两因子地位对等 | ❌ 控制变量 vs 目标变量不对等 |
| 解释含义 | "全市场 Size&B/M 的交集" | "控制 Size 后的 B/M 效应" |
| 适用场景 | 因子相关性低 | 因子相关性高 |

In [ ]:
# ========== 依赖排序 ==========
# 步骤 1: 按 Size 分 5 组 (已完成)
# 步骤 2: 在每个 Size 组内部按 B/M 排序

def conditional_sort(group):
    """在每个 Size 组内部按 B/M 排序分组"""
    n = len(group)
    sorted_group = group.sort_values('B2M').copy()
    # 对于每组只有少量股票的情况，直接按排序赋标签
    labels = [f'B{i+1}' for i in range(n)]
    sorted_group['B2M_Dep_Group'] = labels
    return sorted_group

df_dep = df_demo.groupby('Size_Group', group_keys=False).apply(conditional_sort)

# 展示依赖排序过程
print("📊 依赖排序详细过程：")
print("═" * 70)
for sg in ['S1', 'S2', 'S3', 'S4', 'S5']:
    sub = df_dep[df_dep['Size_Group'] == sg].sort_values('B2M')
    print(f"\n  {sg} 组（市值范围 {sub['Size'].min():.0f}~{sub['Size'].max():.0f} 亿）:")
    print(f"  按 B/M 从小到大排列后赋标签:")
    for _, row in sub.iterrows():
        print(f"    {row['股票']}: B/M={row['B2M']:.3f} → {row['B2M_Dep_Group']}")

print("\n" + "═" * 70)

In [ ]:
# ========== 对比独立 vs 依赖排序 ==========
# 依赖排序的股票分布
count_dep = df_dep.groupby(['Size_Group', 'B2M_Dep_Group']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 独立排序
sns.heatmap(count_matrix, annot=True, fmt='d', cmap='YlOrRd',
             linewidths=1, ax=axes[0], cbar_kws={'label': 'Stock Count'},
            vmin=0, vmax=3)
axes[0].set_title(f'Independent Sort: {n_empty} Empty Cells', fontsize=13, fontweight='bold')
axes[0].set_xlabel('B/M Group')
axes[0].set_ylabel('Size Group')

# 依赖排序
sns.heatmap(count_dep, annot=True, fmt='d', cmap='YlOrRd',
             linewidths=1, ax=axes[1], cbar_kws={'label': 'Stock Count'},
            vmin=0, vmax=3)
n_empty_dep = (count_dep == 0).sum().sum()
axes[1].set_title(f'Conditional Sort: {n_empty_dep} Empty Cells', fontsize=13, fontweight='bold')
axes[1].set_xlabel('B/M Group (Within-Group)')
axes[1].set_ylabel('Size Group')

plt.tight_layout()
plt.show()

print(f"\n💡 对比结论：")
print(f"  独立排序：{n_empty}/{n_total} 个空桶 ({n_empty/n_total*100:.0f}%) → ❌ 分析不完整")
print(f"  依赖排序：{n_empty_dep}/{count_dep.size} 个空桶 ({n_empty_dep/count_dep.size*100:.0f}%) → ✅ 完美填充")

### 4.4 双重排序后的完整计算步骤

完成依赖排序分组后，我们需要：

1. 计算每个 (Size, B/M) 交叉组合的平均收益
2. 在每个 Size 组内构建 B/M 的多空组合
3. 对所有 Size 组的 Spread 取平均 → 得到"控制了 Size 后的纯 B/M 因子溢价"

```
Spread_S1 = R(S1,B5) − R(S1,B1)    # S1 组内的 B/M 溢价
Spread_S2 = R(S2,B5) − R(S2,B1)    # S2 组内的 B/M 溢价
...
Spread_S5 = R(S5,B5) − R(S5,B1)    # S5 组内的 B/M 溢价

Spread_B/M = (Spread_S1 + Spread_S2 + ... + Spread_S5) / 5
           = "控制了 Size 后的纯 B/M 因子溢价"
```

In [ ]:
# ========== 步骤 1: 计算交叉组合收益 ==========
portfolio_rets_dep = df_dep.groupby(['Size_Group', 'B2M_Dep_Group'])['Ret'].mean().unstack()

print("📊 步骤 1: 各 (Size, B/M) 交叉组合的平均收益 (%)")
print("─" * 55)
print(portfolio_rets_dep.round(2))

print("\n💡 每一行是一个 Size 组，每一列是组内 B/M 从低到高")
print("   如果 B/M 因子有效，每行内应该从左到右递增（B1 < B5）")

In [ ]:
# ========== 步骤 2: 在每个 Size 组内构建 B/M 多空组合 ==========
print("📊 步骤 2: 构建 B/M 多空组合（控制 Size）")
print("─" * 60)

size_spreads = {}
for sg in ['S1', 'S2', 'S3', 'S4', 'S5']:
    if 'B5' in portfolio_rets_dep.columns and 'B1' in portfolio_rets_dep.columns:
        r_high = portfolio_rets_dep.loc[sg, 'B5']
        r_low = portfolio_rets_dep.loc[sg, 'B1']
        spread = r_high - r_low
        size_spreads[sg] = spread
        print(f"  {sg}: Spread = R(B5) − R(B1) = {r_high:.2f} − {r_low:.2f} = {spread:.2f}%")

# 总平均 Spread
avg_spread = np.mean(list(size_spreads.values()))
print(f"\n  📐 平均 Spread = ({' + '.join([f'{v:.2f}' for v in size_spreads.values()])}) / {len(size_spreads)}")
print(f"                 = {sum(size_spreads.values()):.2f} / {len(size_spreads)}")
print(f"                 = {avg_spread:.2f}%")
print(f'\n  💡 这就是"控制了 Size 后的纯 B/M 因子溢价" = {avg_spread:.2f}%')

In [ ]:
# ========== 可视化双重排序结果 ==========
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图：收益率热力图 ---
ax1 = axes[0]
sns.heatmap(portfolio_rets_dep, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=1, ax=ax1, cbar_kws={'label': 'Return (%)'})
ax1.set_title('Double Sort (Conditional): Return Matrix', fontsize=13, fontweight='bold')
ax1.set_xlabel('B/M Group (Within-Group)', fontsize=11)
ax1.set_ylabel('Size Group', fontsize=11)

# --- 右图：每个 Size 组内的 Spread ---
ax2 = axes[1]
sg_labels = list(size_spreads.keys())
sg_values = list(size_spreads.values())
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in sg_values]
bars = ax2.bar(sg_labels, sg_values, color=colors, edgecolor='black', alpha=0.85)

# 标注数值
for bar, v in zip(bars, sg_values):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.1 if v >= 0 else -0.1
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
             f'{v:.2f}%', ha='center', va=va, fontweight='bold')

ax2.axhline(y=avg_spread, color='blue', linestyle='--', linewidth=2,
            label=f'Mean Spread = {avg_spread:.2f}%')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Size Group', fontsize=11)
ax2.set_ylabel('B/M Spread (B5-B1) (%)', fontsize=11)
ax2.set_title('B/M Factor Spread by Size Group', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 左图：颜色越绿 = 收益越高。观察每行内是否从左到右变绿（B/M 效应）")
print("   右图：每个 Size 组内 B/M 的 Spread。若全为正值，说明控制 Size 后 B/M 因子仍然有效")

---

## 5. 完整实践：多期双重排序 + T 检验

上一节用 25 只股票展示了机制，但只有 1 期数据，无法做 T 检验。

现在我们**模拟 60 个月的数据**，每月重新双重排序，收集 Spread 时间序列，做正式的 T 检验。

In [ ]:
# ========== 多期双重排序模拟 ==========
np.random.seed(123)
N_STOCKS = 300
N_MONTHS = 60
N_SIZE_GROUPS = 5
N_BM_GROUPS = 5

# 存储每月的 Spread
monthly_bm_spreads = []    # B/M 因子 Spread（控制 Size）

# 存储每月各 B/M 组的收益（跨 Size 组平均）
monthly_bm_group_rets = {f'B{i}': [] for i in range(1, N_BM_GROUPS + 1)}

print(f"🔄 开始多期双重排序 ({N_MONTHS} 个月, {N_STOCKS} 只股票/月)...")

for t in range(N_MONTHS):
    # --- 生成截面数据 ---
    size = np.random.lognormal(mean=10, sigma=1, size=N_STOCKS)
    log_size = np.log(size)

    # B/M 与 Size 负相关
    btm = 1.5 - 0.08 * (log_size - log_size.mean()) + np.random.normal(0, 0.5, N_STOCKS)
    btm = np.clip(btm, 0.1, 3.0)

    # 收益 = 基础 + Size效应(负) + B/M效应(正) + 噪声
    ret = (1.0
           - 0.2 * (log_size - log_size.mean())   # Size 效应
           + 1.5 * (btm - btm.mean())             # B/M 效应
           + np.random.normal(0, 5, N_STOCKS))     # 噪声

    month_df = pd.DataFrame({'size': size, 'btm': btm, 'ret': ret})

    # --- 依赖排序：先按 Size，再在组内按 B/M ---
    month_df['size_g'] = pd.qcut(month_df['size'], N_SIZE_GROUPS,
                                   labels=[f'S{i}' for i in range(1, N_SIZE_GROUPS+1)])

    def cond_sort(g):
        g = g.copy()
        g['bm_g'] = pd.qcut(g['btm'], N_BM_GROUPS,
                              labels=[f'B{i}' for i in range(1, N_BM_GROUPS+1)])
        return g

    month_df = month_df.groupby('size_g', group_keys=False).apply(cond_sort)

    # --- 计算交叉组合收益 ---
    cross_rets = month_df.groupby(['size_g', 'bm_g'])['ret'].mean().unstack()

    # --- B/M Spread (控制 Size) ---
    bm_spread_per_size = cross_rets['B5'] - cross_rets['B1']
    avg_bm_spread = bm_spread_per_size.mean()
    monthly_bm_spreads.append(avg_bm_spread)

    # --- 各 B/M 组收益（跨 Size 平均）---
    for bg in [f'B{i}' for i in range(1, N_BM_GROUPS+1)]:
        monthly_bm_group_rets[bg].append(cross_rets[bg].mean())

    # 打印前 2 个月
    if t < 2:
        print(f"\n  月份 {t+1}: 交叉组合收益矩阵")
        print(cross_rets.round(2).to_string())
        print(f"  → B/M Spread (各Size组平均) = {avg_bm_spread:.2f}%")

print(f"\n  ... (省略第 3~{N_MONTHS} 月) ...")
print(f"\n✅ 完成！共收集 {len(monthly_bm_spreads)} 个月的 B/M Spread")

In [ ]:
# ========== T 检验：B/M 因子溢价 ==========
bm_spreads = np.array(monthly_bm_spreads)

# 逐步计算
bm_mean = bm_spreads.mean()
bm_std = bm_spreads.std(ddof=1)
bm_se = bm_std / np.sqrt(N_MONTHS)
bm_t = bm_mean / bm_se
bm_df = N_MONTHS - 1
bm_p = 2 * (1 - stats.t.cdf(abs(bm_t), df=bm_df))

print("📊 B/M 因子 T 检验（双重排序，控制 Size）")
print("═" * 55)
print(f"  H₀: E[Spread] = 0 (B/M 因子无效)")
print(f"  H₁: E[Spread] ≠ 0 (B/M 因子有效)")
print(f"")
print(f"  Spread 均值 R̄ = {bm_mean:.4f}%")
print(f"  Spread 标准差 σ = {bm_std:.4f}%")
print(f"  标准误 SE = σ/√T = {bm_std:.4f}/√{N_MONTHS} = {bm_se:.4f}%")
print(f"")
print(f"  📐 t = R̄ / SE = {bm_mean:.4f} / {bm_se:.4f} = {bm_t:.4f}")
print(f"  P 值 = {bm_p:.6f}")
print(f"")
if bm_p < 0.05:
    print(f"  ✅ |t| = {abs(bm_t):.2f} > 2，P = {bm_p:.4f} < 0.05")
    print(f"  ✅ 拒绝 H₀：控制 Size 后，B/M 因子溢价显著不为零")
    print(f"  ✅ 结论：B/M (价值因子) 独立于 Size 因子，具有独立的解释力！")
else:
    print(f"  ❌ |t| = {abs(bm_t):.2f}，P = {bm_p:.4f} ≥ 0.05")
    print(f"  ❌ 不能拒绝 H₀")

# scipy 验证
t_v, p_v = stats.ttest_1samp(bm_spreads, 0)
print(f"\n  🔬 scipy 验证: t = {t_v:.4f}, P = {p_v:.6f} ✅")

In [ ]:
# ========== 单调性检验 + 可视化 ==========
# 各 B/M 组的时间序列平均收益
avg_bm_rets = {bg: np.mean(monthly_bm_group_rets[bg]) for bg in monthly_bm_group_rets}

# Spearman 秩相关
bm_ranks = np.arange(1, N_BM_GROUPS + 1)
bm_ret_vals = np.array([avg_bm_rets[f'B{i}'] for i in range(1, N_BM_GROUPS + 1)])
sp_corr, sp_p = stats.spearmanr(bm_ranks, bm_ret_vals)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: 各组收益柱状图 ---
ax1 = axes[0]
bm_labels = [f'B{i}' for i in range(1, N_BM_GROUPS + 1)]
bm_vals = [avg_bm_rets[bg] for bg in bm_labels]
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, N_BM_GROUPS))
bars = ax1.bar(bm_labels, bm_vals, color=colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars, bm_vals):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{v:.3f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax1.plot(bm_labels, bm_vals, 'ko--', linewidth=2, markersize=8, zorder=5)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('B/M Group (B1=Lowest → B5=Highest)', fontsize=11)
ax1.set_ylabel('Average Monthly Return (%)', fontsize=11)
ax1.set_title(f'B/M Monotonicity (\u03c1 = {sp_corr:.3f})', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 图2: Spread 时间序列 ---
ax2 = axes[1]
months = range(1, N_MONTHS + 1)
ax2.bar(months, bm_spreads, color=['#2ecc71' if s > 0 else '#e74c3c' for s in bm_spreads],
        alpha=0.7, edgecolor='none')
ax2.axhline(y=bm_mean, color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {bm_mean:.3f}%')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('B/M Spread (%)', fontsize=11)
ax2.set_title('Monthly B/M Spread (Size-Controlled)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

# --- 图3: T 分布 ---
ax3 = axes[2]
x_t = np.linspace(-5, 5, 1000)
t_dist = stats.t.pdf(x_t, df=bm_df)
ax3.plot(x_t, t_dist, 'b-', linewidth=2, label=f'T distribution (df={bm_df})')
ax3.fill_between(x_t, t_dist, where=(x_t >= abs(bm_t)), color='red', alpha=0.4)
ax3.fill_between(x_t, t_dist, where=(x_t <= -abs(bm_t)), color='red', alpha=0.4,
                 label=f'P = {bm_p:.4f}')
ax3.axvline(x=bm_t, color='red', linestyle='--', linewidth=2, label=f't = {bm_t:.2f}')
ax3.set_xlabel('T Value', fontsize=11)
ax3.set_ylabel('Probability Density', fontsize=11)
ax3.set_title('B/M Factor T-Test', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📋 B/M 因子检验汇总:")
print(f"  Spread 均值: {bm_mean:.3f}%")
print(f"  T 统计量: {bm_t:.2f}")
print(f"  P 值: {bm_p:.4f}")
print(f"  Spearman ρ: {sp_corr:.3f}")

### 📐 95% 置信区间

置信区间帮助我们理解因子溢价估计的**精度**。

$$\text{CI}_{95\%} = \bar{R}_{spread} \pm t_{critical} \times \text{SE}$$

In [ ]:
# ========== 95% 置信区间 ==========
t_critical = stats.t.ppf(0.975, df=bm_df)  # 双尾 2.5%
margin = t_critical * bm_se
ci_lower = bm_mean - margin
ci_upper = bm_mean + margin

print("📊 95% 置信区间")
print("─" * 50)
print(f"  t 临界值 (df={bm_df}, 95%) = {t_critical:.4f}")
print(f"  误差范围 = {t_critical:.4f} × {bm_se:.4f} = {margin:.4f}%")
print(f"")
print(f"  📐 CI = [{ci_lower:.4f}%, {ci_upper:.4f}%]")
print(f"")
if ci_lower > 0:
    print(f"  ✅ 置信区间不包含 0 → 因子溢价显著为正")
    print(f"  ✅ 我们有 95% 的信心，B/M 因子月均溢价在 {ci_lower:.3f}% 到 {ci_upper:.3f}% 之间")
elif ci_upper < 0:
    print(f"  ✅ 置信区间不包含 0 → 因子溢价显著为负")
else:
    print(f"  ❌ 置信区间包含 0 → 因子溢价不显著")

print(f"\n  💡 注意：置信区间与 T 检验结论一致！")
print(f"     如果 P < 0.05，则 95% CI 不包含 0，反之亦然")

### 📐 经济显著性：Spread 的 Sharpe Ratio

统计显著性（$|t| > 2$）告诉我们因子溢价不为零，但**经济显著性**告诉我们这个溢价值不值得去赚。

多空组合的年化 Sharpe Ratio 是衡量经济显著性的核心指标：

$$\text{SR}_{\text{annual}} = \frac{\bar{R}_{\text{spread}}}{\sigma_{\text{spread}}} \times \sqrt{12}$$

经验法则：
- $\text{SR} < 0.5$：经济意义较弱
- $0.5 \leq \text{SR} < 1.0$：经济意义中等
- $\text{SR} \geq 1.0$：经济意义强

注意 T 统计量与年化 Sharpe Ratio 的关系：

$$t = \text{SR}_{\text{monthly}} \times \sqrt{T} = \frac{\text{SR}_{\text{annual}}}{\sqrt{12}} \times \sqrt{T}$$

In [ ]:
# ========== 经济显著性：Sharpe Ratio ==========
sr_monthly = bm_mean / bm_std
sr_annual = sr_monthly * np.sqrt(12)

print("📊 经济显著性分析")
print("─" * 50)
print(f"  月均 Spread: {bm_mean:.4f}%")
print(f"  月均 Spread 标准差: {bm_std:.4f}%")
print(f"  月度 Sharpe Ratio: {sr_monthly:.4f}")
print(f"  年化 Sharpe Ratio: {sr_annual:.4f}")
print(f"")
print(f"  📐 验证: t = SR_monthly × √T = {sr_monthly:.4f} × √{N_MONTHS} = {sr_monthly * np.sqrt(N_MONTHS):.4f}")
print(f"     实际 t = {bm_t:.4f} ✅")
print(f"")
if abs(sr_annual) >= 1.0:
    print(f"  ✅ 年化 SR = {sr_annual:.2f} ≥ 1.0，经济意义强")
elif abs(sr_annual) >= 0.5:
    print(f"  ✅ 年化 SR = {sr_annual:.2f} ∈ [0.5, 1.0)，经济意义中等")
else:
    print(f"  ⚠️ 年化 SR = {sr_annual:.2f} < 0.5，经济意义较弱")
print(f"")
print(f"  💡 统计显著 + 经济显著 = 因子值得关注")
print(f"     统计显著但 SR 很低 → 可能扣除交易成本后无法盈利")

---

## 6. 核心概念回顾

### 📌 投资组合排序法 (Portfolio Sorts)
- **定义**: 按因子值排序分组，构建多空组合检验因子有效性
- **核心**: 排序 → 分组 → 计算组收益 → 构建 Spread
- **目标**: 构建因子模拟投资组合 (FMP)

### 📌 多空组合 (Long-Short Portfolio)
- **定义**: 做多高暴露组，做空低暴露组的零净投入组合
- **公式**: $R_{\text{spread}} = R_{\text{Long}} - R_{\text{Short}}$
- **意义**: 剥离市场风险，提取纯因子收益

### 📌 T 检验
- **定义**: 检验因子溢价是否显著不为零
- **公式**: $t = \bar{R}_{\text{spread}} / (\sigma_{\text{spread}} / \sqrt{T})$
- **判断**: $|t| > 2$ 为显著（经验法则），$|t| > 3$ 更严格

### 📌 单调性检验
- **定义**: 收益率是否随因子暴露单调变化
- **方法**: Spearman 秩相关系数 $\rho_s$
- **判断**: $|\rho_s|$ 接近 1 表示因子质量优秀

### 📌 双重排序
- **独立排序**: 两因子独立分组取交集，适合低相关因子
- **依赖排序**: 先控制变量分组再组内分目标变量，适合高相关因子
- **选择**: 因子相关性高时必须用依赖排序，避免空桶问题

### 📌 经济显著性
- **指标**: 多空组合的年化 Sharpe Ratio
- **公式**: $\text{SR}_{\text{annual}} = (\bar{R}_{\text{spread}} / \sigma_{\text{spread}}) \times \sqrt{12}$
- **意义**: 统计显著不等于经济可行，需考虑交易成本

### 🔗 完整流程
```
数据准备 → 选择排序方式 → 每期执行排序分组
    → 收集 Spread 时间序列 → T 检验 + 单调性检验
    → 置信区间 + 经济显著性 → 综合判断
```

### 📝 检验指标汇总

| 概念 | 要点 | 判断标准 |
|------|------|----------|
| FMP | 通过多空组合"模拟"因子暴露 | Spread = Long − Short |
| T 检验 | 因子溢价是否显著不为零 | $\|t\| > 2$ |
| 单调性 | 收益率是否随因子暴露单调变化 | $\|\rho_s\|$ 接近 1 |
| 独立排序 | 两因子独立分组取交集 | 适合低相关因子 |
| 依赖排序 | 先分控制变量再组内分目标变量 | 适合高相关因子 |
| 置信区间 | 因子溢价的估计精度 | 不含 0 = 显著 |
| Sharpe Ratio | 经济显著性 | SR $\geq$ 0.5 |

### ❌ 常见误区

#### 误区 1: 只看 1 期数据就下结论
**✓ 正确做法**: 必须用多期数据做 T 检验，1 期数据的 Spread 可能纯属偶然

#### 误区 2: 只看 T 检验不看单调性
**✓ 正确做法**: T 检验通过但单调性差 → 可能只是 Q1 和 Q5 差异大，中间组混乱，实践中不可靠

#### 误区 3: 因子相关时用独立排序
**✓ 正确做法**: 因子间相关性高时应该用依赖排序，避免空桶导致结论偏差

#### 误区 4: 把统计显著等同于经济意义
**✓ 正确做法**: $|t| > 2$ 说明统计上显著，但 Spread 的 Sharpe Ratio 才决定经济意义；考虑交易成本后，微小的 Spread 可能无法盈利

#### 误区 5: 忽视数据挖掘偏差
**✓ 正确做法**: Harvey, Liu & Zhu (2016) 建议用 $|t| > 3$ 作为更严格的阈值，以应对多重检验问题